# 04 — RAG over ASIC RG 271

## Purpose

This notebook builds the first retrieval layer for the complaint intelligence platform.

The goal is to use ASIC Regulatory Guide 271 as a small knowledge base for internal dispute resolution guidance.

The workflow is:
- Load the ASIC RG 271 PDF
- Extract text from the PDF
- Split the text into smaller chunks
- Convert chunks into embeddings
- Store embeddings in a local Chroma vector database
- Retrieve the most relevant RG 271 chunks for a redacted complaint narrative

This creates the foundation for a RAG assistant that can support complaint triage with regulatory context.

In [1]:
import pandas as pd
from pathlib import Path
from pypdf import PdfReader

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 300)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PDF_PATH = Path("../data/knowledge_base/rg271.pdf")

print("PDF exists:", PDF_PATH.exists())
print("PDF path:", PDF_PATH)

PDF exists: True
PDF path: ../data/knowledge_base/rg271.pdf


In [3]:
reader = PdfReader(PDF_PATH)

num_pages = len(reader.pages)

print("Number of pages:", num_pages)

Number of pages: 57


In [4]:
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()
    
    if text:
        pages.append({
            "page_number": page_number,
            "text": text
        })

pages_df = pd.DataFrame(pages)

print("Pages with extracted text:", len(pages_df))

pages_df.head()

Pages with extracted text: 57


,page_number,text
0,1,"\n \n \nREGULATORY GUIDE 271 \nInternal dispute resolution \nSeptember 2021 \nAbout this guide \nThis guide is for Australian financial services (AFS) licensees, unlicensed \nproduct issuers, unlicensed secondary sellers, trustees of regulated \nsuperannuation funds (other than self-managed sup..."
1,2,REGULATORY GUIDE 271: Internal dispute resolution \n© Australian Securities and Investments Commission September 2021 Page 2 \nAbout ASIC regulatory documents \nIn administering legislation ASIC issues the following types of regulatory \ndocuments. \nConsultation papers: seek feedback from stak...
2,3,REGULATORY GUIDE 271: Internal dispute resolution \n© Australian Securities and Investments Commission September 2021 Page 3 \nContents \nA Overview ................................................................................................. 4 \nFinancial services dispute resolution framew...
3,4,REGULATORY GUIDE 271: Internal dispute resolution \n© Australian Securities and Investments Commission September 2021 Page 4 \nA Overview \nKey points \nFinancial firms must have a dispute resolution system that consists of: \n• an internal dispute resolution (IDR) procedure that meets the sta...
4,5,"REGULATORY GUIDE 271: Internal dispute resolution \n© Australian Securities and Investments Commission September 2021 Page 5 \nNote 2: Unlicensed carried over instrument lenders (unlicensed COI lenders) have IDR \nobligations, but are not required to be a member of AFCA (see RG 271.3). \nRG 271..."


In [5]:
print(pages_df.loc[0, "text"][:1500])

 
 
 
REGULATORY GUIDE 271 
Internal dispute resolution 
September 2021 
About this guide 
This guide is for Australian financial services (AFS) licensees, unlicensed 
product issuers, unlicensed secondary sellers, trustees of regulated 
superannuation funds (other than self-managed superannuation funds 
(SMSFs)), trustees of approved deposit funds, retirement savings account 
providers, Australian credit licensees (credit licensees) and unlicensed 
carried over instrument lenders (unlicensed COI lenders).  
The standards and requirements highlighted in this guide are enforceable. 
It explains what these financial firms must do to have an internal dispute 
resolution (IDR) system in place that meets ASIC’s standards and 
requirements. 
Note: This guide comes into effect on 5 October 2021. For complaints 
received by financial firms before that date, Regulatory Guide 165 Licensing: 
Internal and external dispute resolution (RG 165) applies. We will withdraw 
RG 165 on 5 October 2022. 
T

In [6]:
full_text = "\n\n".join(pages_df["text"].tolist())

print("Total characters:", len(full_text))
print("Total words:", len(full_text.split()))
print(full_text[:1000])

Total characters: 126909
Total words: 18635
 
 
 
REGULATORY GUIDE 271 
Internal dispute resolution 
September 2021 
About this guide 
This guide is for Australian financial services (AFS) licensees, unlicensed 
product issuers, unlicensed secondary sellers, trustees of regulated 
superannuation funds (other than self-managed superannuation funds 
(SMSFs)), trustees of approved deposit funds, retirement savings account 
providers, Australian credit licensees (credit licensees) and unlicensed 
carried over instrument lenders (unlicensed COI lenders).  
The standards and requirements highlighted in this guide are enforceable. 
It explains what these financial firms must do to have an internal dispute 
resolution (IDR) system in place that meets ASIC’s standards and 
requirements. 
Note: This guide comes into effect on 5 October 2021. For complaints 
received by financial firms before that date, Regulatory Guide 165 Licensing: 
Internal and external dispute resolution (RG 165) applies. We

In [7]:
import re

def clean_pdf_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    # Birden çok boşluk, yeni satır veya sekme karakterini tek bir boşluğa dönüştürür.
    text = text.replace(" .", ".")
    # Noktadan önceki gereksiz boşluğu kaldırır.
    text = text.replace(" ,", ",")
    # Virgülden önceki gereksiz boşluğu kaldırır.
    return text.strip()
    # Metnin başındaki ve sonundaki boşlukları temizler.

clean_text = clean_pdf_text(full_text)

print("Clean characters:", len(clean_text))
print(clean_text[:1000])

clean_text = clean_pdf_text(full_text)

print("Clean characters:", len(clean_text))
print(clean_text[:1000])

Clean characters: 124357
REGULATORY GUIDE 271 Internal dispute resolution September 2021 About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements. Note: This guide comes into effect on 5 October 2021. For complaints received by financial firms before that date, Regulatory Guide 165 Licensing: Internal and external dispute resolution (RG 165) applies. We will withdraw RG 165 on 5 October 2022. T

In [8]:
def recursive_chunk_text(
    text: str,
    chunk_size: int = 1200,
    chunk_overlap: int = 150,
    separators: list[str] = ["\n\n", "\n", ". ", "; ", ", ", " "]
) -> list[str]:
    chunks = []
    # sonuçları saklamak için boş bir liste oluşturulur
    
    def split_recursive(segment: str, seps: list[str]):
        # bir segmenti ayırmak için iç fonksiyon
        if len(segment) <= chunk_size:
            # segment boyutu sınıra uygunsa olduğu gibi döndür
            return [segment.strip()]
        
        if not seps:
            # ayırıcı kalmadıysa düz olarak parça parça ayır
            return [
                segment[i:i + chunk_size].strip()
                for i in range(0, len(segment), chunk_size - chunk_overlap)
            ]
        
        sep = seps[0]
        # ilk ayırıcıyı seç
        parts = segment.split(sep)
        # segmenti seçilen ayırıcıyla böl
        
        temp_chunks = []
        current = ""
        # geçici parça listesi ve birleştirme değişkeni
        
        for part in parts:
            candidate = current + sep + part if current else part
            # mevcut parça ile yeni parçayı birleştir
            
            if len(candidate) <= chunk_size:
                current = candidate
                # eğer birleştirilmiş parça sınıra sığarsa devam et
            else:
                if current:
                    temp_chunks.append(current.strip())
                    # mevcut parçayı kaydet
                current = part
                # yeni parçayı başlat
        
        if current:
            temp_chunks.append(current.strip())
            # döngü sonunda kalan parçayı ekle
        
        final = []
        for chunk in temp_chunks:
            if len(chunk) > chunk_size:
                final.extend(split_recursive(chunk, seps[1:]))
                # parça hala çok büyükse sonraki ayırıcılarla tekrar ayır
            else:
                final.append(chunk)
                # uygun boyuttaysa final listesine ekle
        
        return final
    
    base_chunks = split_recursive(text, separators)
    # metni başlangıç ayırıcılara göre parçala
    
    for i, chunk in enumerate(base_chunks):
        if i == 0:
            chunks.append(chunk)
            # ilk parçayı doğrudan ekle
        else:
            previous_tail = chunks[-1][-chunk_overlap:]
            # önceki parçanın son bölümünü al
            chunks.append((previous_tail + " " + chunk).strip())
            # önceki parça ile örtüşmeyi ekle
    
    return chunks
    # sonuç listesini döndür

recursive_chunks = recursive_chunk_text(clean_text)

print("Recursive chunks:", len(recursive_chunks))
print("First chunk length:", len(recursive_chunks[0]))
print(recursive_chunks[0][:1200])


recursive_chunks = recursive_chunk_text(clean_text)

print("Recursive chunks:", len(recursive_chunks))
print("First chunk length:", len(recursive_chunks[0]))
print(recursive_chunks[0][:1200])

Recursive chunks: 125
First chunk length: 1108
REGULATORY GUIDE 271 Internal dispute resolution September 2021 About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements. Note: This guide comes into effect on 5 October 2021. For complaints received by financial firms before that date, Regulatory Guide 165 Licensing: Internal and external dispute resolution (RG 165) applies. We will withdraw RG 16

In [9]:
for i, chunk in enumerate(recursive_chunks[:5]):
    print(f"\n--- Recursive Chunk {i} ---")
    print("Length:", len(chunk))
    print(chunk[:1000])


--- Recursive Chunk 0 ---
Length: 1108
REGULATORY GUIDE 271 Internal dispute resolution September 2021 About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements. Note: This guide comes into effect on 5 October 2021. For complaints received by financial firms before that date, Regulatory Guide 165 Licensing: Internal and external dispute resolution (RG 165) applies. We will withdraw RG 165 on 5 

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [11]:
def split_into_sentences(text: str) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 30]
    return sentences

sentences = split_into_sentences(clean_text)

print("Number of sentences:", len(sentences))
print(sentences[:5])

Number of sentences: 524
['REGULATORY GUIDE 271 Internal dispute resolution September 2021 About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders).', 'The standards and requirements highlighted in this guide are enforceable.', 'It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements.', 'Note: This guide comes into effect on 5 October 2021.', 'For complaints received by financial firms before that date, Regulatory Guide 165 Licensing: Internal and external dispute resolution (RG 165) applies.']


In [12]:
sentence_embeddings = embedding_model.encode(
    sentences,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Sentence embeddings shape:", sentence_embeddings.shape)


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Sentence embeddings shape: (524, 384)


In [13]:
similarities = []

for i in range(len(sentence_embeddings) - 1):
    sim = np.dot(sentence_embeddings[i], sentence_embeddings[i + 1])
    similarities.append(sim)

similarities = np.array(similarities)

print("Similarity min:", similarities.min())
print("Similarity mean:", similarities.mean())
print("Similarity max:", similarities.max())

Similarity min: 0.015105246
Similarity mean: 0.47839808
Similarity max: 0.9681772


In [14]:
def semantic_chunk_sentences(
    sentences: list[str],
    similarities: np.ndarray,
    percentile_threshold: int = 20,
    max_chunk_chars: int = 1600,
    min_chunk_chars: int = 500
) -> list[str]:
    threshold = np.percentile(similarities, percentile_threshold)
    
    chunks = []
    current_chunk = []
    current_length = 0
    
    for i, sentence in enumerate(sentences):
        current_chunk.append(sentence)
        current_length += len(sentence)
        
        should_split_by_similarity = (
            i < len(similarities) and similarities[i] < threshold
        )
        
        should_split_by_size = current_length >= max_chunk_chars
        
        if (should_split_by_similarity and current_length >= min_chunk_chars) or should_split_by_size:
            chunks.append(" ".join(current_chunk).strip())
            current_chunk = []
            current_length = 0
    
    if current_chunk:
        chunks.append(" ".join(current_chunk).strip())
    
    return chunks


semantic_chunks = semantic_chunk_sentences(
    sentences,
    similarities,
    percentile_threshold=20,
    max_chunk_chars=1600,
    min_chunk_chars=500
)

print("Semantic chunks:", len(semantic_chunks))
print("First semantic chunk length:", len(semantic_chunks[0]))
print(semantic_chunks[0][:1200])

Semantic chunks: 92
First semantic chunk length: 724
REGULATORY GUIDE 271 Internal dispute resolution September 2021 About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements.


In [15]:
for i, chunk in enumerate(semantic_chunks[:5]):
    print(f"\n--- Semantic Chunk {i} ---")
    print("Length:", len(chunk))
    print(chunk[:1000])


--- Semantic Chunk 0 ---
Length: 724
REGULATORY GUIDE 271 Internal dispute resolution September 2021 About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements.

--- Semantic Chunk 1 ---
Length: 1442
Note: This guide comes into effect on 5 October 2021. For complaints received by financial firms before that date, Regulatory Guide 165 Licensing: Internal and external dispute resolution (RG 165) a

In [16]:
## Improved PDF Text Cleaning
import re

def clean_rg271_page_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    
    # Remove common ASIC header/footer noise
    text = re.sub(r"REGULATORY GUIDE 271: Internal dispute resolution", " ", text)
    text = re.sub(r"© Australian Securities and Investments Commission", " ", text)
    text = re.sub(r"September 2021", " ", text)
    text = re.sub(r"Page \d+", " ", text)
    
    # Remove excessive dot leaders from table of contents
    text = re.sub(r"\.{5,}", " ", text)
    
    # Normalize spaces but keep paragraph-ish line breaks first
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    
    return text.strip()


pages_df["clean_page_text"] = pages_df["text"].apply(clean_rg271_page_text)

clean_text_with_breaks = "\n\n".join(pages_df["clean_page_text"].tolist())

print(clean_text_with_breaks[:2000])

REGULATORY GUIDE 271 
Internal dispute resolution 
 
About this guide 
This guide is for Australian financial services (AFS) licensees, unlicensed 
product issuers, unlicensed secondary sellers, trustees of regulated 
superannuation funds (other than self-managed superannuation funds 
(SMSFs)), trustees of approved deposit funds, retirement savings account 
providers, Australian credit licensees (credit licensees) and unlicensed 
carried over instrument lenders (unlicensed COI lenders). 
The standards and requirements highlighted in this guide are enforceable. 
It explains what these financial firms must do to have an internal dispute 
resolution (IDR) system in place that meets ASIC’s standards and 
requirements. 
Note: This guide comes into effect on 5 October 2021. For complaints 
received by financial firms before that date, Regulatory Guide 165 Licensing: 
Internal and external dispute resolution (RG 165) applies. We will withdraw 
RG 165 on 5 October 2022. 
This guide should be r

In [17]:
# Contents kısmını çıkaralım su an chunk’lara “Contents A Overview…” gibi işe yaramaz bölüm giriyor. RAG için bu zayıf context.
def remove_table_of_contents(text: str) -> str:
    # RG 271 contents page usually starts with "Contents" and ends around "A Overview"
    pattern = r"Contents\s+A Overview.*?(?=\nA Overview|\nA\s+Overview)"
    cleaned = re.sub(pattern, " ", text, flags=re.DOTALL)
    return cleaned.strip()

clean_rg271_text = remove_table_of_contents(clean_text_with_breaks)

print("Characters after cleaning:", len(clean_rg271_text))
print(clean_rg271_text[:2000])

Characters after cleaning: 116453
REGULATORY GUIDE 271 
Internal dispute resolution 
 
About this guide 
This guide is for Australian financial services (AFS) licensees, unlicensed 
product issuers, unlicensed secondary sellers, trustees of regulated 
superannuation funds (other than self-managed superannuation funds 
(SMSFs)), trustees of approved deposit funds, retirement savings account 
providers, Australian credit licensees (credit licensees) and unlicensed 
carried over instrument lenders (unlicensed COI lenders). 
The standards and requirements highlighted in this guide are enforceable. 
It explains what these financial firms must do to have an internal dispute 
resolution (IDR) system in place that meets ASIC’s standards and 
requirements. 
Note: This guide comes into effect on 5 October 2021. For complaints 
received by financial firms before that date, Regulatory Guide 165 Licensing: 
Internal and external dispute resolution (RG 165) applies. We will withdraw 
RG 165 on 5 Oct

In [18]:
## Method A — Section-Aware Chunking
def section_aware_chunking(text: str, max_chunk_chars: int = 1800, overlap_chars: int = 150):
    """
    Split RG 271 text using section-like boundaries first, then size-control within each section.
    This is better for regulatory PDFs because it preserves legal / policy structure.
    """
    
    # Common RG 271 section boundaries
    section_pattern = r"(?=\n[A-Z]\s+[A-Z][^\n]{3,80})"
    raw_sections = re.split(section_pattern, text)
    raw_sections = [s.strip() for s in raw_sections if len(s.strip()) > 200]
    
    chunks = []
    
    for section in raw_sections:
        if len(section) <= max_chunk_chars:
            chunks.append(section)
        else:
            # sentence-aware fallback inside long sections
            sentences = re.split(r"(?<=[.!?])\s+", section)
            current = ""
            
            for sentence in sentences:
                candidate = (current + " " + sentence).strip()
                
                if len(candidate) <= max_chunk_chars:
                    current = candidate
                else:
                    if current:
                        chunks.append(current)
                    current = sentence
            
            if current:
                chunks.append(current)
    
    # add light overlap
    overlapped_chunks = []
    for i, chunk in enumerate(chunks):
        if i == 0:
            overlapped_chunks.append(chunk)
        else:
            previous_tail = overlapped_chunks[-1][-overlap_chars:]
            overlapped_chunks.append((previous_tail + " " + chunk).strip())
    
    return overlapped_chunks


section_chunks = section_aware_chunking(clean_rg271_text)

print("Section-aware chunks:", len(section_chunks))
print("First chunk length:", len(section_chunks[0]))
print(section_chunks[0][:1200])


Section-aware chunks: 74
First chunk length: 1679
REGULATORY GUIDE 271 
Internal dispute resolution 
 
About this guide 
This guide is for Australian financial services (AFS) licensees, unlicensed 
product issuers, unlicensed secondary sellers, trustees of regulated 
superannuation funds (other than self-managed superannuation funds 
(SMSFs)), trustees of approved deposit funds, retirement savings account 
providers, Australian credit licensees (credit licensees) and unlicensed 
carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute 
resolution (IDR) system in place that meets ASIC’s standards and 
requirements. Note: This guide comes into effect on 5 October 2021. For complaints 
received by financial firms before that date, Regulatory Guide 165 Licensing: 
Internal and external dispute resolution (RG 165) applies. We will withdraw 
RG

In [19]:
for i, chunk in enumerate(section_chunks[:8]):
    print(f"\n--- Section-aware Chunk {i} ---")
    print("Length:", len(chunk))
    print(chunk[:1200])


--- Section-aware Chunk 0 ---
Length: 1679
REGULATORY GUIDE 271 
Internal dispute resolution 
 
About this guide 
This guide is for Australian financial services (AFS) licensees, unlicensed 
product issuers, unlicensed secondary sellers, trustees of regulated 
superannuation funds (other than self-managed superannuation funds 
(SMSFs)), trustees of approved deposit funds, retirement savings account 
providers, Australian credit licensees (credit licensees) and unlicensed 
carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute 
resolution (IDR) system in place that meets ASIC’s standards and 
requirements. Note: This guide comes into effect on 5 October 2021. For complaints 
received by financial firms before that date, Regulatory Guide 165 Licensing: 
Internal and external dispute resolution (RG 165) applies. We will withdraw 
RG 165 o

In [20]:
## Method B — Improved Semantic Chunking
def split_into_sentences(text: str) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 40]
    return sentences

sentences = split_into_sentences(clean_rg271_text)

print("Number of sentences:", len(sentences))
print(sentences[:5])


Number of sentences: 521
['REGULATORY GUIDE 271 \nInternal dispute resolution \n \nAbout this guide \nThis guide is for Australian financial services (AFS) licensees, unlicensed \nproduct issuers, unlicensed secondary sellers, trustees of regulated \nsuperannuation funds (other than self-managed superannuation funds \n(SMSFs)), trustees of approved deposit funds, retirement savings account \nproviders, Australian credit licensees (credit licensees) and unlicensed \ncarried over instrument lenders (unlicensed COI lenders).', 'The standards and requirements highlighted in this guide are enforceable.', 'It explains what these financial firms must do to have an internal dispute \nresolution (IDR) system in place that meets ASIC’s standards and \nrequirements.', 'Note: This guide comes into effect on 5 October 2021.', 'For complaints \nreceived by financial firms before that date, Regulatory Guide 165 Licensing: \nInternal and external dispute resolution (RG 165) applies.']


In [21]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

sentence_embeddings = embedding_model.encode(
    sentences,
    show_progress_bar=True,
    normalize_embeddings=True
)

similarities = np.array([
    np.dot(sentence_embeddings[i], sentence_embeddings[i + 1])
    for i in range(len(sentence_embeddings) - 1)
])

print("Similarity min:", similarities.min())
print("Similarity mean:", similarities.mean())
print("Similarity max:", similarities.max())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Similarity min: 0.015105246
Similarity mean: 0.48143506
Similarity max: 0.9681772


In [22]:
def improved_semantic_chunking(
    sentences: list[str],
    similarities: np.ndarray,
    percentile_threshold: int = 25,
    max_chunk_chars: int = 1400,
    min_chunk_chars: int = 450
) -> list[str]:
    
    threshold = np.percentile(similarities, percentile_threshold)
    
    chunks = []
    current_chunk = []
    current_length = 0
    
    for i, sentence in enumerate(sentences):
        current_chunk.append(sentence)
        current_length += len(sentence)
        
        semantic_break = i < len(similarities) and similarities[i] < threshold
        size_break = current_length >= max_chunk_chars
        
        if (semantic_break and current_length >= min_chunk_chars) or size_break:
            chunks.append(" ".join(current_chunk).strip())
            current_chunk = []
            current_length = 0
    
    if current_chunk:
        chunks.append(" ".join(current_chunk).strip())
    
    return chunks


semantic_chunks_v2 = improved_semantic_chunking(
    sentences,
    similarities,
    percentile_threshold=25,
    max_chunk_chars=1400,
    min_chunk_chars=450
)

print("Improved semantic chunks:", len(semantic_chunks_v2))
print("First chunk length:", len(semantic_chunks_v2[0]))
print(semantic_chunks_v2[0][:1200])

Improved semantic chunks: 101
First chunk length: 565
REGULATORY GUIDE 271 
Internal dispute resolution 
 
About this guide 
This guide is for Australian financial services (AFS) licensees, unlicensed 
product issuers, unlicensed secondary sellers, trustees of regulated 
superannuation funds (other than self-managed superannuation funds 
(SMSFs)), trustees of approved deposit funds, retirement savings account 
providers, Australian credit licensees (credit licensees) and unlicensed 
carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable.


In [23]:
for i, chunk in enumerate(semantic_chunks_v2[:8]):
    print(f"\n--- Semantic v2 Chunk {i} ---")
    print("Length:", len(chunk))
    print(chunk[:1200])


--- Semantic v2 Chunk 0 ---
Length: 565
REGULATORY GUIDE 271 
Internal dispute resolution 
 
About this guide 
This guide is for Australian financial services (AFS) licensees, unlicensed 
product issuers, unlicensed secondary sellers, trustees of regulated 
superannuation funds (other than self-managed superannuation funds 
(SMSFs)), trustees of approved deposit funds, retirement savings account 
providers, Australian credit licensees (credit licensees) and unlicensed 
carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable.

--- Semantic v2 Chunk 1 ---
Length: 1491
It explains what these financial firms must do to have an internal dispute 
resolution (IDR) system in place that meets ASIC’s standards and 
requirements. Note: This guide comes into effect on 5 October 2021. For complaints 
received by financial firms before that date, Regulatory Guide 165 Licensing: 
Internal and external dispute resolution (RG 1

In [24]:
chunk_comparison_v2 = pd.DataFrame([
    {
        "method": "recursive_v1",
        "num_chunks": len(recursive_chunks),
        "avg_chunk_chars": round(np.mean([len(c) for c in recursive_chunks]), 1),
        "min_chunk_chars": min(len(c) for c in recursive_chunks),
        "max_chunk_chars": max(len(c) for c in recursive_chunks),
        "manual_quality": "poor - cuts across structure"
    },
    {
        "method": "section_aware",
        "num_chunks": len(section_chunks),
        "avg_chunk_chars": round(np.mean([len(c) for c in section_chunks]), 1),
        "min_chunk_chars": min(len(c) for c in section_chunks),
        "max_chunk_chars": max(len(c) for c in section_chunks),
        "manual_quality": "expected better for regulatory PDF"
    },
    {
        "method": "semantic_v2",
        "num_chunks": len(semantic_chunks_v2),
        "avg_chunk_chars": round(np.mean([len(c) for c in semantic_chunks_v2]), 1),
        "min_chunk_chars": min(len(c) for c in semantic_chunks_v2),
        "max_chunk_chars": max(len(c) for c in semantic_chunks_v2),
        "manual_quality": "expected better for topic shifts"
    }
])

chunk_comparison_v2

,method,num_chunks,avg_chunk_chars,min_chunk_chars,max_chunk_chars,manual_quality
0,recursive_v1,125,1142.5,223,1351,poor - cuts across structure
1,section_aware,74,1716.8,444,1949,expected better for regulatory PDF
2,semantic_v2,101,1148.3,452,3162,expected better for topic shifts


## Method C — Paragraph / Clause-Aware Chunking

This method is designed for regulatory documents such as ASIC RG 271.

Instead of splitting only by character length or semantic distance, it tries to preserve paragraph and clause structure.

The goal is to create chunks that:
- do not cut words or sentences in the middle,
- preserve regulatory context,
- avoid very small orphan chunks,
- avoid very large mixed-topic chunks,
- remain readable as standalone retrieval units.

In [25]:
def extract_paragraphs(text: str) -> list[str]:
    """
    Extract paragraph-like blocks from cleaned PDF text.
    Keeps meaningful blocks and removes very short/noisy fragments.
    """
    raw_blocks = re.split(r"\n\s*\n", text)
    
    paragraphs = []
    
    for block in raw_blocks:
        block = block.strip()
        block = re.sub(r"\s+", " ", block)
        
        # remove very short fragments
        if len(block) < 80:
            continue
        
        # remove obvious table-of-contents style fragments
        if "Contents" in block and "Overview" in block and "Application" in block:
            continue
        
        paragraphs.append(block)
    
    return paragraphs


paragraphs = extract_paragraphs(clean_rg271_text)

print("Number of paragraphs:", len(paragraphs))

for i, p in enumerate(paragraphs[:5]):
    print(f"\n--- Paragraph {i} ---")
    print("Length:", len(p))
    print(p[:800])

Number of paragraphs: 57

--- Paragraph 0 ---
Length: 1045
About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements. Note: This guide comes into effect on 5 October 2021. For complaints received by financial firms before that date, Regulatory Guide 165 Lic

--- Paragraph 1 ---
Length: 1553
About ASIC regulatory documents In administering legislation ASIC issues the following types of regulatory

In [26]:
def split_long_paragraph(paragraph: str, max_chars: int = 1600) -> list[str]:
    """
    Split a long paragraph into sentence-aware pieces.
    """
    if len(paragraph) <= max_chars:
        return [paragraph]
    
    sentences = re.split(r"(?<=[.!?])\s+", paragraph)
    
    pieces = []
    current = ""
    
    for sentence in sentences:
        candidate = (current + " " + sentence).strip()
        
        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                pieces.append(current)
            current = sentence
    
    if current:
        pieces.append(current)
    
    return pieces


print("Long paragraph splitter ready.")

Long paragraph splitter ready.


In [27]:
def paragraph_aware_chunking(
    paragraphs: list[str],
    min_chunk_chars: int = 600,
    max_chunk_chars: int = 1800,
    overlap_chars: int = 120
) -> list[str]:
    """
    Build chunks by combining paragraphs until the chunk is large enough,
    without exceeding max length too much.
    """
    chunks = []
    current_chunk = ""
    
    for paragraph in paragraphs:
        # If paragraph is too long, split it first
        paragraph_pieces = split_long_paragraph(paragraph, max_chars=max_chunk_chars)
        
        for piece in paragraph_pieces:
            candidate = (current_chunk + "\n\n" + piece).strip() if current_chunk else piece
            
            if len(candidate) <= max_chunk_chars:
                current_chunk = candidate
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = piece
        
        if len(current_chunk) >= min_chunk_chars:
            chunks.append(current_chunk.strip())
            current_chunk = ""
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    # Add light overlap, but only at character level for continuity
    overlapped_chunks = []
    
    for i, chunk in enumerate(chunks):
        if i == 0:
            overlapped_chunks.append(chunk)
        else:
            previous_tail = overlapped_chunks[-1][-overlap_chars:]
            overlapped_chunks.append((previous_tail + "\n\n" + chunk).strip())
    
    return overlapped_chunks


paragraph_chunks = paragraph_aware_chunking(paragraphs)

print("Paragraph-aware chunks:", len(paragraph_chunks))
print("First chunk length:", len(paragraph_chunks[0]))
print(paragraph_chunks[0][:1200])

Paragraph-aware chunks: 90
First chunk length: 1045
About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements. Note: This guide comes into effect on 5 October 2021. For complaints received by financial firms before that date, Regulatory Guide 165 Licensing: Internal and external dispute resolution (RG 165) applies. We will withdraw RG 165 on 5 October 2022. This guide should be read in conjuncti

In [28]:
for i, chunk in enumerate(paragraph_chunks[:8]):
    print(f"\n--- Paragraph-aware Chunk {i} ---")
    print("Length:", len(chunk))
    print(chunk[:1200])


--- Paragraph-aware Chunk 0 ---
Length: 1045
About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements. Note: This guide comes into effect on 5 October 2021. For complaints received by financial firms before that date, Regulatory Guide 165 Licensing: Internal and external dispute resolution (RG 165) applies. We will withdraw RG 165 on 5 October 2022. This guide should be read in conjunction wit

In [29]:
chunk_comparison_v3 = pd.DataFrame([
    {
        "method": "recursive_v1",
        "num_chunks": len(recursive_chunks),
        "avg_chunk_chars": round(np.mean([len(c) for c in recursive_chunks]), 1),
        "min_chunk_chars": min(len(c) for c in recursive_chunks),
        "max_chunk_chars": max(len(c) for c in recursive_chunks),
        "manual_quality": "poor - cuts across structure"
    },
    {
        "method": "section_aware",
        "num_chunks": len(section_chunks),
        "avg_chunk_chars": round(np.mean([len(c) for c in section_chunks]), 1),
        "min_chunk_chars": min(len(c) for c in section_chunks),
        "max_chunk_chars": max(len(c) for c in section_chunks),
        "manual_quality": "good candidate for regulatory PDF"
    },
    {
        "method": "semantic_v2",
        "num_chunks": len(semantic_chunks_v2),
        "avg_chunk_chars": round(np.mean([len(c) for c in semantic_chunks_v2]), 1),
        "min_chunk_chars": min(len(c) for c in semantic_chunks_v2),
        "max_chunk_chars": max(len(c) for c in semantic_chunks_v2),
        "manual_quality": "mixed - better than recursive but sometimes too broad"
    },
    {
        "method": "paragraph_aware",
        "num_chunks": len(paragraph_chunks),
        "avg_chunk_chars": round(np.mean([len(c) for c in paragraph_chunks]), 1),
        "min_chunk_chars": min(len(c) for c in paragraph_chunks),
        "max_chunk_chars": max(len(c) for c in paragraph_chunks),
        "manual_quality": "expected strongest for readable retrieval chunks"
    }
])

chunk_comparison_v3

,method,num_chunks,avg_chunk_chars,min_chunk_chars,max_chunk_chars,manual_quality
0,recursive_v1,125,1142.5,223,1351,poor - cuts across structure
1,section_aware,74,1716.8,444,1949,good candidate for regulatory PDF
2,semantic_v2,101,1148.3,452,3162,mixed - better than recursive but sometimes too broad
3,paragraph_aware,90,1389.1,211,1922,expected strongest for readable retrieval chunks


## Method D — RG Paragraph-Aware Chunking

The previous methods produced readable chunks in some places, but still created poor boundaries because PDF text extraction and character-level overlap caused chunks to begin in the middle of words or sentences.

For ASIC RG 271, a better approach is to use the document's own regulatory paragraph structure.

This method splits the text around `RG 271.x` paragraph references, then groups nearby paragraphs into readable retrieval chunks.

The goal is to create chunks that:
- start at meaningful regulatory paragraph boundaries,
- preserve RG paragraph references,
- avoid broken word starts,
- avoid character-level overlap,
- remain useful as standalone RAG context.

In [30]:
import re
import pandas as pd
import numpy as np

def extract_rg_paragraph_blocks(text: str) -> list[dict]:
    """
    Extract blocks starting with RG 271.x references.
    Each block keeps its RG paragraph number and related text.
    """
    pattern = r"(RG\s+271\.\d+[A-Z]?)"
    
    matches = list(re.finditer(pattern, text))
    blocks = []
    
    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        
        rg_ref = match.group(1)
        block_text = text[start:end].strip()
        block_text = re.sub(r"\s+", " ", block_text).strip()
        
        if len(block_text) > 80:
            blocks.append({
                "rg_ref": rg_ref,
                "text": block_text,
                "length": len(block_text)
            })
    
    return blocks


rg_blocks = extract_rg_paragraph_blocks(clean_rg271_text)

print("RG paragraph blocks:", len(rg_blocks))

pd.DataFrame(rg_blocks).head(10)

RG paragraph blocks: 230


,rg_ref,text,length
0,RG 271.1,"RG 271.1 Financial firms must have in place a dispute resolution system that consists of: (a) an IDR procedure that complies with standards and requirements made or approved by ASIC; and (b) membership of AFCA. Note 1: See s912A(1)(g) and 1017G(1) of the Corporations Act 2001 (Corporations Act),...",691
1,RG 271.2,"RG 271.2 Most financial firms also have a requirement to comply with their IDR procedures: see modified s912A(1)(g) and 1017G(1) of the Corporations Act, and modified s47(1)(h) and (i) of the National Credit Act.",212
2,RG 271.3,"RG 271.3 A modified regulatory regime applies to some unlicensed credit firms. Credit representatives and exempt special purpose funding entities (exempt SPFEs) (including securitisation bodies) do not have IDR obligations, but must be a member of AFCA. Unlicensed carried over instrument lenders...",697
3,RG 271.4,RG 271.4 Table 1 sets out the dispute resolution requirements by type of financial firm. Table 1: Legislative dispute resolution requirements by firm type Firm type Description Dispute resolution requirements Australian financial services (AFS) licensees An AFS licensee is a business carrying ou...,10625
4,RG 271.5,"RG 271.5 The objectives of Ch 7 of the Corporations Act are to promote: (a) the confident and informed participation of consumers and investors in the Australian financial system (also an objective of ASIC under s1 of the Australian Securities and Investments Commission Act 2001); (b) fairness, ...",488
5,RG 271.6,"RG 271.6 Within this framework, we are responsible for overseeing the effective operation of the dispute resolution system, which includes setting the standards and requirements for financial firms’ IDR processes and oversight of AFCA.",235
6,RG 271.7,"RG 271.7 We must, when considering whether to make or approve standards or requirements relating to IDR, take into account: (a) AS/NZS 10002:2014; and Note: AS/NZS 10002:2014 is published by SAI Global and available for purchase on their website. It is also available through public libraries acr...",538
7,RG 271.8,"RG 271.8 The standards and requirements set out in ASIC Corporations, Credit and Superannuation (Internal Dispute Resolution) Instrument 2020/98 and highlighted in this guide are enforceable. Other highlighted requirements in this guide reflect existing legal requirements and are also enforceable.",298
8,RG 271.9,RG 271.9 The parts of this guide that we have not highlighted or set out in the instrument are guidance to help financial firms comply with their legal obligations.,164
9,RG 271.10,"RG 271.10 We may vary or revoke: (a) a standard or requirement that we have made for IDR; and (b) the operation of a standard or requirement that we have approved in its application to IDR. Note: See regs 7.6.02(2) and 7.9.77(2) of the Corporations Regulations, and reg 10(2) and item 2.20 of Sch...",334


In [31]:
for i, block in enumerate(rg_blocks[:8]):
    print(f"\n--- RG Block {i} | {block['rg_ref']} | Length: {block['length']} ---")
    print(block["text"][:1000])


--- RG Block 0 | RG 271.1 | Length: 691 ---
RG 271.1 Financial firms must have in place a dispute resolution system that consists of: (a) an IDR procedure that complies with standards and requirements made or approved by ASIC; and (b) membership of AFCA. Note 1: See s912A(1)(g) and 1017G(1) of the Corporations Act 2001 (Corporations Act), s47(1)(h) and (i) of the National Consumer Credit Protection Act 2009 (National Credit Act), s101(1) and (1A) of the Superannuation Industry (Supervision) Act 1993 (SIS Act), and s47(1) and (2) of the Retirement Savings Account Act 1997 (RSA Act). Note 2: Unlicensed carried over instrument lenders (unlicensed COI lenders) have IDR obligations, but are not required to be a member of AFCA (see

--- RG Block 1 | RG 271.2 | Length: 212 ---
RG 271.2 Most financial firms also have a requirement to comply with their IDR procedures: see modified s912A(1)(g) and 1017G(1) of the Corporations Act, and modified s47(1)(h) and (i) of the National Credit Act.

--- 

In [32]:
def build_rg_aware_chunks(
    rg_blocks: list[dict],
    min_chunk_chars: int = 700,
    max_chunk_chars: int = 1800
) -> list[dict]:
    """
    Combine RG paragraph blocks into retrieval chunks.
    No character-level overlap is used to avoid broken starts.
    """
    chunks = []
    current_texts = []
    current_refs = []
    current_length = 0
    
    for block in rg_blocks:
        block_text = block["text"]
        block_ref = block["rg_ref"]
        
        candidate_length = current_length + len(block_text) + 2
        
        if candidate_length <= max_chunk_chars:
            current_texts.append(block_text)
            current_refs.append(block_ref)
            current_length = candidate_length
        else:
            if current_texts:
                chunks.append({
                    "chunk_id": len(chunks),
                    "method": "rg_paragraph_aware",
                    "rg_refs": current_refs.copy(),
                    "text": "\n\n".join(current_texts),
                    "length": len("\n\n".join(current_texts))
                })
            
            current_texts = [block_text]
            current_refs = [block_ref]
            current_length = len(block_text)
        
        if current_length >= min_chunk_chars:
            chunks.append({
                "chunk_id": len(chunks),
                "method": "rg_paragraph_aware",
                "rg_refs": current_refs.copy(),
                "text": "\n\n".join(current_texts),
                "length": len("\n\n".join(current_texts))
            })
            current_texts = []
            current_refs = []
            current_length = 0
    
    if current_texts:
        chunks.append({
            "chunk_id": len(chunks),
            "method": "rg_paragraph_aware",
            "rg_refs": current_refs.copy(),
            "text": "\n\n".join(current_texts),
            "length": len("\n\n".join(current_texts))
        })
    
    return chunks


rg_chunks = build_rg_aware_chunks(rg_blocks)

print("RG-aware chunks:", len(rg_chunks))

rg_chunks_df = pd.DataFrame(rg_chunks)
rg_chunks_df.head()

RG-aware chunks: 88


,chunk_id,method,rg_refs,text,length
0,0,rg_paragraph_aware,"[RG 271.1, RG 271.2]","RG 271.1 Financial firms must have in place a dispute resolution system that consists of: (a) an IDR procedure that complies with standards and requirements made or approved by ASIC; and (b) membership of AFCA. Note 1: See s912A(1)(g) and 1017G(1) of the Corporations Act 2001 (Corporations Act),...",905
1,1,rg_paragraph_aware,[RG 271.3],"RG 271.3 A modified regulatory regime applies to some unlicensed credit firms. Credit representatives and exempt special purpose funding entities (exempt SPFEs) (including securitisation bodies) do not have IDR obligations, but must be a member of AFCA. Unlicensed carried over instrument lenders...",697
2,2,rg_paragraph_aware,[RG 271.4],RG 271.4 Table 1 sets out the dispute resolution requirements by type of financial firm. Table 1: Legislative dispute resolution requirements by firm type Firm type Description Dispute resolution requirements Australian financial services (AFS) licensees An AFS licensee is a business carrying ou...,10625
3,3,rg_paragraph_aware,"[RG 271.5, RG 271.6]","RG 271.5 The objectives of Ch 7 of the Corporations Act are to promote: (a) the confident and informed participation of consumers and investors in the Australian financial system (also an objective of ASIC under s1 of the Australian Securities and Investments Commission Act 2001); (b) fairness, ...",725
4,4,rg_paragraph_aware,"[RG 271.7, RG 271.8]","RG 271.7 We must, when considering whether to make or approve standards or requirements relating to IDR, take into account: (a) AS/NZS 10002:2014; and Note: AS/NZS 10002:2014 is published by SAI Global and available for purchase on their website. It is also available through public libraries acr...",838


In [33]:
for i, row in rg_chunks_df.head(10).iterrows():
    print(f"\n--- RG-aware Chunk {row['chunk_id']} ---")
    print("RG refs:", row["rg_refs"])
    print("Length:", row["length"])
    print(row["text"][:1200])


--- RG-aware Chunk 0 ---
RG refs: ['RG 271.1', 'RG 271.2']
Length: 905
RG 271.1 Financial firms must have in place a dispute resolution system that consists of: (a) an IDR procedure that complies with standards and requirements made or approved by ASIC; and (b) membership of AFCA. Note 1: See s912A(1)(g) and 1017G(1) of the Corporations Act 2001 (Corporations Act), s47(1)(h) and (i) of the National Consumer Credit Protection Act 2009 (National Credit Act), s101(1) and (1A) of the Superannuation Industry (Supervision) Act 1993 (SIS Act), and s47(1) and (2) of the Retirement Savings Account Act 1997 (RSA Act). Note 2: Unlicensed carried over instrument lenders (unlicensed COI lenders) have IDR obligations, but are not required to be a member of AFCA (see

RG 271.2 Most financial firms also have a requirement to comply with their IDR procedures: see modified s912A(1)(g) and 1017G(1) of the Corporations Act, and modified s47(1)(h) and (i) of the National Credit Act.

--- RG-aware Chunk 1 

## Method E — Agentic Chunking with LLM

The previous chunking methods were not satisfactory for this regulatory PDF because they still produced chunks with broken starts, weak paragraph boundaries, or overly broad context.

This method uses an LLM to create standalone, meaning-preserving regulatory chunks.

The chunking rules are:
- Do not split words.
- Do not split sentences.
- Preserve headings and RG references.
- Keep each chunk self-contained.
- Split long tables or clauses into meaningful subparts.
- Return structured JSON for downstream RAG ingestion.

In [34]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import pandas as pd
import re
from pathlib import Path

# Find .env by walking up from the current working directory.
env_path = None
for base in [Path.cwd(), *Path.cwd().parents]:
    candidate = base / ".env"
    if candidate.exists():
        env_path = candidate
        break

if env_path:
    load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        f"OPENAI_API_KEY not found. Checked current working directory chain starting from: {Path.cwd()}"
    )

client = OpenAI(api_key=api_key)

print("OpenAI client created.")
print("Loaded .env from:", env_path)
print("API key available:", bool(api_key))


OpenAI client created.
Loaded .env from: /Users/osmanorka/Complaint-Intelligence-Platform-PII-Safe-Classification-RAG-Assistant-for-Financial-Complaints/complaint-intelligence-platform/.env
API key available: True


In [35]:
AGENTIC_CHUNKING_SYSTEM_PROMPT = """
You are an expert regulatory document chunking assistant.

Your task is to split ASIC RG 271 text into high-quality retrieval chunks for a RAG system.

Rules:
1. Do not split words.
2. Do not split sentences.
3. Preserve headings, section names, RG references, and legal references.
4. Each chunk must be understandable on its own.
5. If a table is too long, split it by logical row groups, not by character count.
6. Keep chunks focused on one topic or regulatory requirement.
7. Do not rewrite the meaning of the text.
8. Do not add legal advice.
9. Remove obvious PDF noise such as page numbers, repeated headers, and copyright footers.
10. Return valid JSON only.

Output JSON format:
{
  "chunks": [
    {
      "title": "short descriptive title",
      "rg_refs": ["RG 271.1"],
      "chunk_type": "overview | requirement | definition | timeframe | table | standard | systemic_issue | other",
      "text": "clean standalone chunk text"
    }
  ]
}
"""

In [36]:
test_agentic_input = rg_chunks_df.loc[0:2, "text"].str.cat(sep="\n\n")

print("Input characters:", len(test_agentic_input))
print(test_agentic_input[:2000])

Input characters: 12231
RG 271.1 Financial firms must have in place a dispute resolution system that consists of: (a) an IDR procedure that complies with standards and requirements made or approved by ASIC; and (b) membership of AFCA. Note 1: See s912A(1)(g) and 1017G(1) of the Corporations Act 2001 (Corporations Act), s47(1)(h) and (i) of the National Consumer Credit Protection Act 2009 (National Credit Act), s101(1) and (1A) of the Superannuation Industry (Supervision) Act 1993 (SIS Act), and s47(1) and (2) of the Retirement Savings Account Act 1997 (RSA Act). Note 2: Unlicensed carried over instrument lenders (unlicensed COI lenders) have IDR obligations, but are not required to be a member of AFCA (see

RG 271.2 Most financial firms also have a requirement to comply with their IDR procedures: see modified s912A(1)(g) and 1017G(1) of the Corporations Act, and modified s47(1)(h) and (i) of the National Credit Act.

RG 271.3 A modified regulatory regime applies to some unlicensed cred

In [37]:
def agentic_chunk_text(text: str, model: str = "gpt-4.1-mini") -> dict:
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": AGENTIC_CHUNKING_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"Chunk the following ASIC RG 271 text:\n\n{text}"
            }
        ],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)


agentic_result = agentic_chunk_text(test_agentic_input)

agentic_result

{'chunks': [{'title': 'Overview of dispute resolution system requirements',
   'rg_refs': ['RG 271.1'],
   'chunk_type': 'requirement',
   'text': 'Financial firms must have in place a dispute resolution system that consists of: (a) an IDR procedure that complies with standards and requirements made or approved by ASIC; and (b) membership of AFCA. Note 1: See s912A(1)(g) and 1017G(1) of the Corporations Act 2001 (Corporations Act), s47(1)(h) and (i) of the National Consumer Credit Protection Act 2009 (National Credit Act), s101(1) and (1A) of the Superannuation Industry (Supervision) Act 1993 (SIS Act), and s47(1) and (2) of the Retirement Savings Account Act 1997 (RSA Act). Note 2: Unlicensed carried over instrument lenders (unlicensed COI lenders) have IDR obligations, but are not required to be a member of AFCA.'},
  {'title': 'Compliance with IDR procedures by financial firms',
   'rg_refs': ['RG 271.2'],
   'chunk_type': 'requirement',
   'text': 'Most financial firms also have a 

In [38]:
agentic_chunks_sample_df = pd.DataFrame(agentic_result["chunks"])

print("Agentic sample chunks:", len(agentic_chunks_sample_df))

agentic_chunks_sample_df

Agentic sample chunks: 9


,title,rg_refs,chunk_type,text
0,Overview of dispute resolution system requirements,[RG 271.1],requirement,"Financial firms must have in place a dispute resolution system that consists of: (a) an IDR procedure that complies with standards and requirements made or approved by ASIC; and (b) membership of AFCA. Note 1: See s912A(1)(g) and 1017G(1) of the Corporations Act 2001 (Corporations Act), s47(1)(h..."
1,Compliance with IDR procedures by financial firms,[RG 271.2],requirement,"Most financial firms also have a requirement to comply with their IDR procedures: see modified s912A(1)(g) and 1017G(1) of the Corporations Act, and modified s47(1)(h) and (i) of the National Credit Act."
2,Modified regulatory regime for unlicensed credit firms,[RG 271.3],requirement,"A modified regulatory regime applies to some unlicensed credit firms. Credit representatives and exempt special purpose funding entities (exempt SPFEs) (including securitisation bodies) do not have IDR obligations, but must be a member of AFCA. Unlicensed carried over instrument lenders (unlicen..."
3,Dispute resolution requirements by firm type - Part 1,[RG 271.4],table,Table 1 sets out the dispute resolution requirements by type of financial firm.\n\nFirm type: Australian financial services (AFS) licensees\nDescription: An AFS licensee is a business carrying out financial services. This includes businesses that: provide financial product advice to clients; dea...
4,Dispute resolution requirements by firm type - Part 2,[RG 271.4],table,"Firm type: Superannuation trustees\nDescription: A trustee of a regulated superannuation fund (other than a self-managed superannuation fund (SMSF)), trustee of an approved deposit fund or a retirement savings account (RSA) provider.\nDispute resolution requirements: Superannuation trustees must..."
5,Dispute resolution requirements by firm type - Part 3,[RG 271.4],table,Firm type: Credit representatives\nDescription: A credit representative is a person authorised to engage in specified credit activities on behalf of a credit licensee under s64(2) or 65(2) of the National Credit Act. The employees and directors of a credit licensee do not need to be formally aut...
6,Dispute resolution requirements by firm type - Part 4,[RG 271.4],table,"Firm type: Unlicensed COI lenders (including prescribed unlicensed COI lenders)\nDescription: A ‘carried over instrument’ is a contract or other instrument that was made and in force, and to which an old Credit Code applied immediately before 1 July 2010 (see s4(1) of the National Consumer Credi..."
7,Dispute resolution requirements by firm type - Part 5,[RG 271.4],table,Firm type: Exempt SPFEs\nDescription: Special purpose funding entities (SPFEs) include securitisation entities and fundraising special purpose entities that make (or buy) loans or leases and repackage them as investment products to sell to investors. Note: See the definition of ‘special purpose ...
8,Dispute resolution requirements for fintech businesses relying on ERS exemption,[RG 271.4],requirement,Financial technology (fintech) businesses relying on the enhanced regulatory sandbox (ERS) exemption provided by Corporations (FinTech Sandbox Australian Financial Services Licence Exemption) Regulations 2020 and National Consumer Credit Protection (FinTech Sandbox Australian Credit Licence Exem...


In [39]:
for i, row in agentic_chunks_sample_df.iterrows():
    print(f"\n--- Agentic Chunk {i} ---")
    print("Title:", row["title"])
    print("RG refs:", row["rg_refs"])
    print("Type:", row["chunk_type"])
    print("Length:", len(row["text"]))
    print(row["text"][:1500])


--- Agentic Chunk 0 ---
Title: Overview of dispute resolution system requirements
RG refs: ['RG 271.1']
Type: requirement
Length: 678
Financial firms must have in place a dispute resolution system that consists of: (a) an IDR procedure that complies with standards and requirements made or approved by ASIC; and (b) membership of AFCA. Note 1: See s912A(1)(g) and 1017G(1) of the Corporations Act 2001 (Corporations Act), s47(1)(h) and (i) of the National Consumer Credit Protection Act 2009 (National Credit Act), s101(1) and (1A) of the Superannuation Industry (Supervision) Act 1993 (SIS Act), and s47(1) and (2) of the Retirement Savings Account Act 1997 (RSA Act). Note 2: Unlicensed carried over instrument lenders (unlicensed COI lenders) have IDR obligations, but are not required to be a member of AFCA.

--- Agentic Chunk 1 ---
Title: Compliance with IDR procedures by financial firms
RG refs: ['RG 271.2']
Type: requirement
Length: 203
Most financial firms also have a requirement to comp

## Agentic Chunking Result

The agentic chunking method produced the strongest chunk quality.

Compared with recursive, semantic, paragraph-aware, and RG-number-aware approaches, the agentic method better preserved regulatory meaning, RG references, headings, and sentence boundaries.

The resulting chunks are readable as standalone retrieval units and include useful metadata such as title, RG references, and chunk type.

For this project, agentic chunking is selected as the primary chunking strategy. Long table-based chunks will still be reviewed and split into logical subparts when needed.